# AI-Powered Credit Card Fraud Detection System

**Course:** AI in Banking & Finance (AIBF) — Unit 3 & Unit 4 Assignment
**Use case:** Fraud Prevention using AI (Unit 4) + Explainable AI, Trust & Ethics (Unit 3)

## Objective
This notebook builds a simple but complete AI-based fraud detection application for the banking
domain. It covers:

1. Loading and exploring real-world, anonymized credit card transaction data
2. Handling severe class imbalance (fraud is rare)
3. Training and comparing multiple AI models for fraud classification
4. Evaluating models with metrics suited to imbalanced fraud data (Precision, Recall, F1, ROC-AUC, PR-AUC)
5. Adding **Explainable AI (XAI)** using SHAP, so a bank analyst can see *why* a transaction was flagged
6. A simple **real-time fraud alert simulator** module that scores new transactions and issues risk levels

This directly maps to the syllabus topics: *Fraud Prevention using AI*, *Explainable AI*, *Trust and Ethics
in AI*, and *Risk Management using AI*.


## 1. Dataset

**Dataset used:** *Credit Card Fraud Detection* (ULB Machine Learning Group)
**Link:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

- 284,807 transactions made by European cardholders in September 2013
- 492 frauds (~0.17% of all transactions) — highly imbalanced, realistic for banking fraud
- Features `V1`–`V28` are PCA-anonymized (for customer privacy — ties into *Privacy issues in Banking
  and Finance*, Unit 3), plus `Time`, `Amount`, and the target `Class` (1 = fraud, 0 = genuine)

### How to get the data

**Option A — Kaggle Notebook (recommended, matches your usual Kaggle T4 workflow):**
1. Open a new Kaggle Notebook
2. Click **Add Input** → search **"Credit Card Fraud Detection"** (by `mlg-ulb`) → Add
3. The file will be available at `/kaggle/input/creditcardfraud/creditcard.csv`
4. Set `DATA_PATH` in the cell below to that path

**Option B — Local machine (Anaconda / VS Code):**
1. Download `creditcard.csv` from the Kaggle link above (requires a free Kaggle account)
2. Place it inside a `dataset/` folder next to this notebook
3. Set `DATA_PATH = "dataset/creditcard.csv"`

**Option C — kagglehub (no manual download, needs Kaggle API credentials configured):**
```python
import kagglehub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print(path)
```


In [ ]:
# ---- Configuration ----
# Change this path depending on where you placed the dataset (see markdown above)
DATA_PATH = "dataset/creditcard.csv"   # local default
# DATA_PATH = "/kaggle/input/creditcardfraud/creditcard.csv"   # uncomment if running on Kaggle

RANDOM_STATE = 42


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

from imblearn.over_sampling import SMOTE

import time
from tqdm.auto import tqdm  # auto-picks notebook widget bar or plain text bar

sns.set_style("whitegrid")
print("Libraries loaded successfully.")


## 2. Load and Explore the Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()
df.describe().T


In [ ]:
# Class imbalance check
class_counts = df["Class"].value_counts()
print(class_counts)
print("\nFraud percentage: {:.4f}%".format(100 * class_counts[1] / len(df)))

plt.figure(figsize=(5, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.xticks([0, 1], ["Genuine (0)", "Fraud (1)"])
plt.title("Class Distribution — Genuine vs Fraudulent Transactions")
plt.ylabel("Count")
plt.show()


In [ ]:
# Transaction amount distribution: fraud vs genuine
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df.Class == 0]["Amount"], bins=50, ax=axes[0], color="steelblue")
axes[0].set_title("Genuine Transaction Amounts")
sns.histplot(df[df.Class == 1]["Amount"], bins=50, ax=axes[1], color="crimson")
axes[1].set_title("Fraudulent Transaction Amounts")
plt.tight_layout()
plt.show()


## 3. Preprocessing

- Scale `Time` and `Amount` (the `V1`-`V28` columns are already PCA-scaled)
- Split into train/test sets *before* balancing, to avoid data leakage
- Apply **SMOTE** (Synthetic Minority Oversampling) only on the training set to handle the
  extreme class imbalance — a standard technique in AI-based fraud/risk systems


In [ ]:
scaler = StandardScaler()
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Time_scaled"] = scaler.fit_transform(df[["Time"]])

feature_cols = [c for c in df.columns if c not in ["Time", "Amount", "Class"]]
X = df[feature_cols]
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train fraud count:", y_train.sum(), " Test fraud count:", y_test.sum())


In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE :", pd.Series(y_train_res).value_counts().to_dict())


## 4. Train AI Models

We train two commonly used models for fraud/credit-risk classification and compare them:

1. **Logistic Regression** — interpretable baseline, widely used in real banking risk scorecards
2. **Random Forest** — a stronger ensemble model that usually improves recall on fraud cases


In [ ]:
# ---- Logistic Regression: verbose training so you can see solver progress ----
print("Training Logistic Regression...")
t0 = time.time()

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, verbose=1)
log_reg.fit(X_train_res, y_train_res)

print(f"Logistic Regression trained in {time.time() - t0:.2f} seconds.\n")

# ---- Random Forest: build trees incrementally (warm_start) with a progress bar ----
print("Training Random Forest...")
t0 = time.time()

TOTAL_TREES = 200
STEP = 20  # add this many trees per update

rf = RandomForestClassifier(
    n_estimators=STEP,
    max_depth=12,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    warm_start=True,   # lets us keep adding trees without retraining from scratch
)

for n_trees in tqdm(range(STEP, TOTAL_TREES + 1, STEP), desc="Random Forest trees"):
    rf.n_estimators = n_trees
    rf.fit(X_train_res, y_train_res)

print(f"Random Forest trained in {time.time() - t0:.2f} seconds ({rf.n_estimators} trees).")
print("Models trained.")


## 5. Evaluate the Models

For fraud detection, **accuracy is misleading** (99.8% accuracy is trivial by predicting "genuine"
every time). We instead focus on **Precision, Recall, F1-score, ROC-AUC, and PR-AUC**, which matter
for **Risk Management using AI**.


In [ ]:
def evaluate_model(model, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred, target_names=["Genuine", "Fraud"]))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("PR-AUC :", round(average_precision_score(y_test, y_proba), 4))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Genuine", "Fraud"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

    return y_proba

proba_lr = evaluate_model(log_reg, "Logistic Regression")


In [ ]:
proba_rf = evaluate_model(rf, "Random Forest")


In [ ]:
# ROC curve comparison
plt.figure(figsize=(6, 5))
for proba, name in [(proba_lr, "Logistic Regression"), (proba_rf, "Random Forest")]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Model Comparison")
plt.legend()
plt.show()


## 6. Explainable AI (XAI)

Regulators and banking risk teams require that AI fraud decisions be explainable — a "black box"
score is not acceptable for compliance or customer disputes. Here we use **SHAP (SHapley Additive
exPlanations)** to explain individual predictions of the Random Forest model, directly addressing
the **Explainable AI** and **Trust and Ethics in AI** topics from Unit 3.


In [ ]:
import shap

# Small background sample for speed
explainer = shap.TreeExplainer(rf)
sample_X = X_test.sample(200, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(sample_X)


def get_fraud_class_shap(shap_values, class_index=1):
    """
    Return SHAP values for the fraud class, regardless of the SHAP library version's
    output format. Older SHAP versions return a list [class0_values, class1_values];
    newer versions (0.44+) return a single array of shape (samples, features, classes).
    """
    if isinstance(shap_values, list):
        return shap_values[class_index]
    arr = np.array(shap_values)
    if arr.ndim == 3:
        return arr[:, :, class_index]
    return arr


shap_values_fraud = get_fraud_class_shap(shap_values, class_index=1)
print("SHAP values shape:", shap_values_fraud.shape, " | sample_X shape:", sample_X.shape)

# Global feature importance (which features drive fraud predictions overall)
shap.summary_plot(shap_values_fraud, sample_X, show=True)


In [ ]:
# Explain a single flagged fraud case
fraud_indices = y_test[y_test == 1].index

if len(fraud_indices) == 0:
    print("No fraud cases in the test set to explain — skipping this cell.")
else:
    example_idx = fraud_indices[0]
    example_row = X_test.loc[[example_idx]]

    pred_proba = rf.predict_proba(example_row)[0][1]
    print(f"Transaction index {example_idx} — Predicted fraud probability: {pred_proba:.4f}")

    single_shap_values = explainer.shap_values(example_row)
    single_shap_fraud = get_fraud_class_shap(single_shap_values, class_index=1)

    # expected_value can be a scalar or an array of per-class base values depending on
    # the SHAP version, so handle both cases the same way as above
    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)) and np.ndim(expected_value) > 0:
        expected_value_fraud = expected_value[1]
    else:
        expected_value_fraud = expected_value

    # Modern Explanation-based waterfall plot (recommended by SHAP over the older,
    # less stable force_plot(matplotlib=True) API used previously)
    explanation = shap.Explanation(
        values=single_shap_fraud[0],
        base_values=expected_value_fraud,
        data=example_row.iloc[0].values,
        feature_names=list(example_row.columns),
    )
    shap.plots.waterfall(explanation, show=True)


## 7. Real-Time Fraud Alert Simulator (Application Module)

This is the "application" part of the assignment: a small reusable function that mimics how a bank's
transaction-monitoring system would use this AI model in production — scoring a transaction and
issuing a **risk level and recommended action**, instead of a raw probability.


In [ ]:
def score_transaction(transaction_row, model=rf, low=0.3, high=0.7):
    """
    Score a single transaction (as a DataFrame row with the same feature columns as X)
    and return a risk assessment, similar to a bank's real-time fraud monitoring system.
    """
    proba = model.predict_proba(transaction_row)[0][1]

    if proba < low:
        risk_level, action = "LOW", "Approve transaction"
    elif proba < high:
        risk_level, action = "MEDIUM", "Flag for manual review"
    else:
        risk_level, action = "HIGH", "Block transaction & alert customer"

    return {
        "fraud_probability": round(float(proba), 4),
        "risk_level": risk_level,
        "recommended_action": action,
    }


# Demo: run the simulator on a few random test transactions
demo_samples = X_test.sample(5, random_state=1)
for idx, row in demo_samples.iterrows():
    result = score_transaction(row.to_frame().T)
    actual = "FRAUD" if y_test.loc[idx] == 1 else "genuine"
    print(f"Transaction {idx} (actual: {actual}) -> {result}")


## 8. Summary

This notebook demonstrates a small but complete AI-based FinTech application for **fraud prevention
in banking**:

- Real, privacy-anonymized transaction data (Privacy issues in Banking & Finance)
- Class-imbalance handling with SMOTE (a practical challenge in Risk Management using AI)
- Two AI models compared using fraud-appropriate metrics (Fraud Prevention / Risk Management using AI)
- SHAP-based explainability for individual decisions (Explainable AI, Trust and Ethics in AI)
- A reusable `score_transaction()` module simulating a real-time bank fraud alert system
  (Banking and Finance Process Automation using AI)

### Possible extensions (for later deliverables / report)
- Wrap `score_transaction()` in a FastAPI/Flask endpoint for a live demo
- Add a Streamlit dashboard for analysts to review flagged transactions
- Log model decisions for an audit trail (supports Trust & Ethics / regulatory compliance)
- Add drift monitoring to retrain the model as fraud patterns evolve
